# **Imports**

In [1]:
%pip install pandas

import numpy as np
import pandas as pd


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\marti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# **Preprocesamiento**

## Detección de Outliers

In [2]:
C = [3, 1, 0, 2, 7, 3, 6, 4, -2, 0, 0, 10, 15, 6]

###Encontrando outliers por medio de desviaciones estándar
def outliers_std(std):
    print(f"Calculando outliers con +- {std} desviaciones estándar...")
    outliers = []
    mean = np.mean(C)
    std_dev = np.std(C)
    print(f"Media: {mean}, Desviación estándar: {std_dev}")
    for value in C:
        if abs(value - mean) > std * std_dev:
            print(f"    Valor {value} es un outlier ({abs(value - mean)} > {std* std_dev})")
            outliers.append(value)
        else:
            print(f"    Valor {value} no es un outlier ({abs(value - mean)} <= {std* std_dev})")
    return outliers

In [3]:
#+- 3 desviaciones estándar
outliers = outliers_std(3)
print("Outliers (3 std):", outliers)

#+-2 Desviaciones estándar
outliers_2std = outliers_std(2)
print("Outliers (2 std):", outliers_2std)

#+-1 Desviación estándar
outliers_1std = outliers_std(1)
print("Outliers (1 std):", outliers_1std)

Calculando outliers con +- 3 desviaciones estándar...
Media: 3.9285714285714284, Desviación estándar: 4.4153027030073115
    Valor 3 no es un outlier (0.9285714285714284 <= 13.245908109021935)
    Valor 1 no es un outlier (2.9285714285714284 <= 13.245908109021935)
    Valor 0 no es un outlier (3.9285714285714284 <= 13.245908109021935)
    Valor 2 no es un outlier (1.9285714285714284 <= 13.245908109021935)
    Valor 7 no es un outlier (3.0714285714285716 <= 13.245908109021935)
    Valor 3 no es un outlier (0.9285714285714284 <= 13.245908109021935)
    Valor 6 no es un outlier (2.0714285714285716 <= 13.245908109021935)
    Valor 4 no es un outlier (0.07142857142857162 <= 13.245908109021935)
    Valor -2 no es un outlier (5.928571428571429 <= 13.245908109021935)
    Valor 0 no es un outlier (3.9285714285714284 <= 13.245908109021935)
    Valor 0 no es un outlier (3.9285714285714284 <= 13.245908109021935)
    Valor 10 no es un outlier (6.071428571428571 <= 13.245908109021935)
    Valor 15 n

## Normalización

In [4]:
def min_max_normalization(array):
    print("Calculando normalización Min-Max...")
    array_min = np.min(array)
    array_max = np.max(array)
    print(f"Min: {array_min}, Max: {array_max}")
    normalized_array = (array - array_min) / (array_max - array_min)
    return np.array(normalized_array)

def z_score_normalization(array):
    print("Calculando normalización Z-Score...")
    mean = np.mean(array)
    std_dev = np.std(array)
    print(f"Media: {mean}, Desviación estándar: {std_dev}")
    z_normalized_array = (array - mean) / std_dev if std_dev != 0 else array - mean
    return np.array(z_normalized_array)

In [5]:
# Normalización Min-Max
array = [20, 23, 25, 26, 29, 34, 35, 37, 39, 40]
print("Array original:", array, "\n")
normalized_array = min_max_normalization(array)
print("Array normalizado (Min-Max):", normalized_array, "\n")
# Normalización Z-Score
z_normalized_array = z_score_normalization(array)
print("Array normalizado (Z-Score):", z_normalized_array)

Array original: [20, 23, 25, 26, 29, 34, 35, 37, 39, 40] 

Calculando normalización Min-Max...
Min: 20, Max: 40
Array normalizado (Min-Max): [0.   0.15 0.25 0.3  0.45 0.7  0.75 0.85 0.95 1.  ] 

Calculando normalización Z-Score...
Media: 30.8, Desviación estándar: 6.749814812274482
Array normalizado (Z-Score): [-1.6000439  -1.15558726 -0.85928283 -0.71113062 -0.26667398  0.47408708
  0.62223929  0.91854372  1.21484814  1.36300036]


## Discretización

### *Técnica BIN*

In [6]:
I1 = [1, 2, 1, 2, 1, 1, 2]
I2 = [5.9, 2.1, 1.6, 6.8, 3.1, 8.3, 2.4] # Uso de la Media como representante
I3 = [3.4, 6.2, 2.8, 5.8, 3.1, 4.1, 5.0] # Uso del Límite más cercano como Representante

In [7]:
def BIN_error(original, binned):
    error = 0
    for o, b in zip(original, binned):
        error += (o - b) ** 2
    return error

In [8]:
I1 = np.sort(I1)
I2 = np.sort(I2)
print("I1:", I1)
print("I2:", I2)

I1: [1 1 1 1 2 2 2]
I2: [1.6 2.1 2.4 3.1 5.9 6.8 8.3]


In [ ]:
def find_best_BIN(original, representative='mean'):
    original = np.sort(original)
    best_error = float('inf')
    best_binned = None
    bin_index = 1
    for i in range(1, len(original) + 1):
        g1 = original[:i]
        g2 = original[i:]
        if representative == 'mean':
            binned = [np.mean(g1)] * len(g1) + [np.mean(g2)] * len(g2)
        elif representative == 'nearest':
            binned = [g1[0]] * len(g1) + [g2[-1]] * len(g2)
        error = BIN_error(original, binned)

        binned_clean = [float(x) for x in binned]

        if error < best_error:
            best_error = error
            best_binned = binned
            bin_index = i
    print(f"Mejor BIN encontrado en el índice {bin_index} y error {best_error}")
    print(f"Original: {[float(x) for x in original]}")
    print(f"Mejor BIN: {[float(x) for x in best_binned]}")
    print("Grupos:")
    print(f"Grupo 1: {[float(x) for x in original[:bin_index]]}")
    print(f"Grupo 2: {[float(x) for x in original[bin_index:]]}")
    return best_binned, best_error, bin_index

In [11]:
print("\nEvaluando I2:")
find_best_BIN(I2)



Evaluando I2:
Mejor BIN encontrado en el índice 4 y error 4.120000000000001
Original: [1.6, 2.1, 2.4, 3.1, 5.9, 6.8, 8.3]
Mejor BIN: [2.3, 2.3, 2.3, 2.3, 7.0, 7.0, 7.0]
Grupos:
Grupo 1: [1.6, 2.1, 2.4, 3.1]
Grupo 2: [5.9, 6.8, 8.3]


([np.float64(2.3),
  np.float64(2.3),
  np.float64(2.3),
  np.float64(2.3),
  np.float64(7.0),
  np.float64(7.0),
  np.float64(7.0)],
 np.float64(4.120000000000001),
 4)